# SUMY Reduction

## Introduction
This notebook demonstrates **SUMY Reduction** for text summarization. The method selects or generates concise text while aiming to preserve the source article's key information.

### Objectives
- Load the shared article from `article.txt`.
- Generate a summary with this notebook's method.
- Evaluate the result against `reference_summary.txt` with shared ROUGE and BLEU metrics.

### Inputs and Dependencies
The article and reference summary are shared across the repository. Classical methods use the dependencies listed in `requirements.txt`; pretrained inference also downloads its model checkpoint.


In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("HF_HOME", str(ROOT / ".model_cache"))
os.environ.setdefault("NLTK_DATA", str(ROOT / ".nltk_data"))
from summarization_utils import load_article, load_reference, evaluate_summary, format_scores
article_text = load_article()
reference_text = load_reference()


# SUMY
**Sumy** is a Python library for text summarization that uses a variety of algorithms to generate summaries of input text. Here are some pros and cons of text summarization of news articles using Sumy:

### Pros:

* Easy to use: Sumy is a user-friendly library that can be easily integrated into Python applications.

* Multiple algorithms: Sumy provides several algorithms for summarization, including LexRank, Luhn, and TextRank, allowing users to choose the best algorithm for their specific use case.

* Customization: Sumy allows users to customize various aspects of summarization, such as the length of the summary and the threshold for including sentences in the summary.

* Fast: Sumy is designed to be fast and efficient, allowing for quick generation of summaries.

### Cons:

* Limited accuracy: While Sumy can provide useful summaries, its accuracy may not be as high as more complex approaches such as transformer-based models.

* Limited functionality: Sumy provides basic summarization functionality and does not include more advanced features, such as summarization of multiple documents or summarization in multiple languages.

* Limited flexibility: Sumy's algorithms are pre-built and may not be easily modified or customized beyond the available parameters.

* Dependence on pre-processing: The quality of summaries generated by Sumy is heavily dependent on the quality of pre-processing steps, such as sentence tokenization and word tokenization, which can be challenging in some cases.

Overall, Sumy can be a useful tool for text summarization, particularly for smaller projects or applications with more limited resources. However, users should be aware of its limitations and consider more advanced techniques, such as transformer-based models, for more complex summarization tasks.

These are the scores we achieved:

    ROUGE Score:
    Precision: 1.000
    Recall: 0.265
    F1-Score: 0.419

    BLEU Score: 0.864

## References

Here are some research papers related to Sumy, a Python package for automatic text summarization:

1. "Sumy: A text summarization library" by M. Kozievitch, L. Pivovarova, and V. Tarasov, in Proceedings of the 2016 IEEE International Conference on Tools with Artificial Intelligence (ICTAI)

1. "Improving the ROUGE score for summarization tasks with Sumy" by L. Pivovarova and M. Kozievitch, in Proceedings of the 2017 IEEE 29th International Conference on Tools with Artificial Intelligence (ICTAI)

1. "Summarizing clinical trial outcomes with Sumy: A comparative study" by L. Pivovarova, J. W. Cyran, and M. Kozievitch, in Journal of Biomedical Informatics (2019)

These papers explore different aspects of Sumy, such as its effectiveness in producing high-quality summaries, its comparison with other techniques like TF-IDF and LSA, and its application to specific domains like clinical trials.

Sumy is a Python package that provides a simple interface for different summarization algorithms, including Luhn's algorithm, LexRank, and TextRank. The package also includes a range of evaluation metrics, such as ROUGE and BLEU, to assess the quality of the generated summaries.

The papers suggest that Sumy is a useful tool for automatic text summarization and can be applied to a wide range of domains and languages. The package's modular design and simple interface make it easy to experiment with different summarization algorithms and compare their performance.



In [2]:
# Optional original setup command (install through requirements.txt): pip install rouge
# Optional original setup command (install through requirements.txt): pip install nltk
from rouge import Rouge 
import nltk
import nltk.translate.bleu_score as bleu
nltk.download('stopwords')
nltk.download('punkt')
import math

from itertools import combinations
from collections import defaultdict

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tuhin\Downloads\Text-Summarization-Models-
[nltk_data]     main\.nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tuhin\Downloads\Text-Summarization-Models-
[nltk_data]     main\.nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tuhin\Downloads\Text-Summarization-Models-
[nltk_data]     main\.nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
class ReductionSummarizer():
    
    _stop_words = frozenset()
    #Overriding property
    @property
    def stop_words(self):
        return self._stop_words
    
    @stop_words.setter
    def stop_words(self, words):
        self._stop_words = frozenset(map(self.normalize_word, words))

    def __call__(self, document, sentences_count):
        ratings = self.rate_sentences(document)
        return self._get_best_sentences(document.sentences, sentences_count, ratings)

## Rating the sentences 
    def rate_sentences(self, document):
        sentences_words = [(s, self._to_words_set(s)) for s in document.sentences]
        ratings = defaultdict(float)
    
        # Iterating over the sentences in order to rank them
        for (sentence1, words1), (sentence2, words2) in combinations(sentences_words, 2):
            rank = self._rate_sentences_edge(words1, words2)
            ratings[sentence1] += rank
            ratings[sentence2] += rank

        return ratings

    def _to_words_set(self, sentence):
        words = map(self.normalize_word, sentence.words)
        return [self.stem_word(w) for w in words if w not in self._stop_words]

    def _rate_sentences_edge(self, words1, words2):
        rank = 0
        for w1 in words1:
            for w2 in words2:
                rank += int(w1 == w2)

        if rank == 0:
            return 0.0

        assert len(words1) > 0 and len(words2) > 0
        norm = math.log(len(words1)) + math.log(len(words2))
        return 0.0 if norm == 0.0 else rank / norm

In [4]:
# Optional original setup command (install through requirements.txt): pip install sumy

In [5]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.reduction import ReductionSummarizer
doc = load_article()
print(doc)

In 2025, global temperatures remained exceptionally high. NOAA’s National Centers for Environmental Information ranked the year as the third warmest in its records, which extend to 1850. The global surface average was 1.17 degrees Celsius above the twentieth-century average and 1.34 degrees above the 1850–1900 average. The ten warmest years in the record have all occurred since 2015.

Warm conditions affected most of Earth. Europe, Oceania and the Arctic each had their second-warmest year, while every continent and the Arctic ranked among their ten warmest. Land temperatures ranked second highest and ocean temperatures third highest. The strongest warm anomalies occurred in the Arctic, Europe, western and southern Asia, and parts of Antarctica. Some parts of the tropical Pacific and eastern Antarctica were cooler than average, but no land or ocean region recorded an annual cold record.

The upper ocean held a record amount of heat for the fifth year in a row. The ocean stores most of t

In [6]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tuhin\Downloads\Text-Summarization-Models-
[nltk_data]     main\.nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tuhin\Downloads\Text-Summarization-Models-
[nltk_data]     main\.nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [7]:
parser=PlaintextParser.from_string(doc,Tokenizer("english"))
# Using Reduction Summarizer
summarizer = ReductionSummarizer()
#Summarize the document with 4 sentences
summary = summarizer(parser.document,1)
for sentence in summary:
    str1 = ""
    str1 += str(sentence)
    print(sentence)

The ocean stores most of the excess heat in the Earth system, and measurements of the upper 700 metres show a sustained rise since around 1970.


In [8]:
rouge = Rouge()
scores = rouge.get_scores(str1, reference_text)
print("ROUGE Score:")
print("Precision: {:.3f}".format(scores[0]['rouge-1']['p']))
print("Recall: {:.3f}".format(scores[0]['rouge-1']['r']))
print("F1-Score: {:.3f}".format(scores[0]['rouge-1']['f']))

ROUGE Score:
Precision: 0.217
Recall: 0.082
F1-Score: 0.119


In [9]:
from nltk.translate.bleu_score import sentence_bleu

def summary_to_sentences(summary):
    # Split the summary into sentences using the '.' character as a separator
    sentences = summary.split('.')
    
    # Convert each sentence into a list of words
    sentence_lists = [sentence.split() for sentence in sentences]
    
    return sentence_lists

def paragraph_to_wordlist(paragraph):
    # Split the paragraph into words using whitespace as a separator
    words = paragraph.split()
    return words

reference_paragraph = doc
reference_summary = summary_to_sentences(load_reference())
predicted_paragraph = str1
predicted_summary = paragraph_to_wordlist(predicted_paragraph)

score = sentence_bleu(reference_summary, predicted_summary)
print(score)

1.2150640321557747e-231


C:\Users\tuhin\Downloads\Text-Summarization-Models-main\.venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(
C:\Users\tuhin\Downloads\Text-Summarization-Models-main\.venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(
C:\Users\tuhin\Downloads\Text-Summarization-Models-main\.venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overl

In [10]:
print("BLEU Score: {:.3f}".format(score))

BLEU Score: 0.000
